In [1]:
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge

from sqlalchemy import select, func
from sqlalchemy.orm import joinedload

from benchmarklib import BenchmarkDatabase
from rbf import RandomBooleanFunction, RandomBooleanFunctionTrial


In [2]:
db = BenchmarkDatabase("rbf.db", RandomBooleanFunction, RandomBooleanFunctionTrial)

In [3]:
def compute_cx_depth(circuit: QuantumCircuit) -> int:
    """
    Heuristic for multiplicative depth: count the depth of only the 2-qubit gates
    """
    return circuit.depth(filter_function=lambda op: op.operation.num_qubits >= 2)

In [4]:
problems = db.query(select(RandomBooleanFunction).limit(10000).order_by(func.random()))
data = []
for problem in problems:
    trials = db.query(
        select(RandomBooleanFunctionTrial)
        .where(RandomBooleanFunctionTrial.problem_id == problem.id, RandomBooleanFunctionTrial._circuit_qpy != None)
        .where(RandomBooleanFunctionTrial.compiler_name == "XAG")  # only use XAG since we care about multiplicative depth
        .options(joinedload(RandomBooleanFunctionTrial.problem))
        .limit(1)
        )
    if len(trials) == 0:
        continue
    trial = trials[0]
    circuit = trial.circuit
    cx_depth = compute_cx_depth(circuit)
    total_depth = circuit.depth()
    num_2q_gates = circuit.count_ops().get("ecr", 0) or circuit.count_ops().get("cx", 0)
    data.append((problem.num_vars, cx_depth, total_depth, num_2q_gates, trial.calculate_success_rate()))

### Regression

In [11]:
# regression predict fidelity from cx_depth and num_2q_gates
X = np.array([[d[1], d[3]] for d in data])  # cx_depth and num_2q_gates
y = np.array([d[4] for d in data])  # fidelity
model = LinearRegression()
model.fit(X, y)
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
print("R^2 score:", model.score(X, y))

Coefficients: [-0.0026481   0.00077315]
Intercept: 0.45242100988366085
R^2 score: 0.36120419363686884


In [8]:
# regression with negative weights
X = np.array([[d[1], d[3]] for d in data])  # cx_depth and num_2q_gates
y = np.array([d[4] for d in data])  # fidelity

# Fit (-X, y) with positive coefficients, which is equivalent to fitting (X, y) with negative coefficients
model = LinearRegression(positive=True)
model.fit(-X, y)

print("Coefficients:", - model.coef_)
print("Intercept:", model.intercept_)
print("R^2 score:", model.score(-X, y))

Coefficients: [-0.00065771 -0.        ]
Intercept: 0.3875360824283611
R^2 score: 0.2675342579194022


In [9]:
# regression with negative weights (including depth)
X = np.array([[d[1], d[3], d[2]] for d in data])  # cx_depth and num_2q_gates and total_depth
y = np.array([d[4] for d in data])  # fidelity

# Fit (-X, y) with positive coefficients, which is equivalent to fitting (X, y) with negative coefficients
model = LinearRegression(positive=True)
model.fit(-X, y)

print("Coefficients:", - model.coef_)
print("Intercept:", model.intercept_)
print("R^2 score:", model.score(-X, y))

Coefficients: [-0.         -0.         -0.00017596]
Intercept: 0.39000678418322426
R^2 score: 0.2707219387793739


In [14]:
# try predicting using only cx_depth
X = np.array([[d[1]] for d in data])  # cx_depth only
model = LinearRegression()
model.fit(X, y)
print("R^2 score (cx_depth only):", model.score(X, y))

R^2 score (cx_depth only): 0.2633967621122155


In [15]:
# try predicting using only num_2q_gates
X = np.array([[d[3]] for d in data])  # num_2q_gates only
model = LinearRegression()
model.fit(X, y)
print("R^2 score (num_2q_gates only):", model.score(X, y))

R^2 score (num_2q_gates only): 0.19623749648390598


### Plot Bins

In [30]:
# create bins on total cx count and for each bin, plot fidelity vs cx_depth
bins = {
    "[1,9]": [],
    "[10,99]": [],
    "[100,999]": [],
    "[1000,9999]": [],
    "[10000,99999]": [],
    "[100000,999999]": [],
}
for num_vars, cx_depth, total_depth, num_2q_gates, fidelity in data:
    if num_2q_gates <= 9:
        bins["[1,9]"].append((cx_depth, fidelity))
    elif num_2q_gates <= 99:
        bins["[10,99]"].append((cx_depth, fidelity))
    elif num_2q_gates <= 999:
        bins["[100,999]"].append((cx_depth, fidelity))
    elif num_2q_gates <= 9999:
        bins["[1000,9999]"].append((cx_depth, fidelity))
    elif num_2q_gates <= 99999:
        bins["[10000,99999]"].append((cx_depth, fidelity))
    elif num_2q_gates <= 999999:
        bins["[100000,999999]"].append((cx_depth, fidelity))

In [ ]:
for bin_name, values in bins.items():
    if len(values) == 0:
        continue
    cx_depth, fidelities = zip(*values)
    plt.figure()
    plt.scatter(cx_depth, fidelities)
    plt.xticks(range(min(cx_depth), max(cx_depth)+1, max(1, max(cx_depth)//10)))
    plt.xlabel("CX Depth")
    plt.ylabel("Fidelity")
    plt.title(f"Fidelity vs CX Depth - Total CX Count in {bin_name}")
    plt.show()